# Jupiter to automize clustering for Cats

In [1]:
import pandas as pd
import numpy as np
import sklearn.metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import autosklearn.classification

## I. Open the datasets

In [2]:
df = pd.read_csv('../data/OutCatdata.csv', na_filter= False)
df = df.drop("Unnamed: 0", axis= 1)

/home/jan_codage/miniforge3/envs/fouille/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3552: DtypeWarning: Columns (11) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


/!\\ the NA values are dropped /!\\

## first look at the data

In [ ]:
df

### Split the datasets between classes to predict and data

In [4]:
variables = df.drop(['Hunt'], axis = 1)
variables

,event.id,timestamp,location.long,location.lat,animal.id,animal.life.stage,animal.reproductive.condition,animal.sex,N.pray,Hrs.indors,N.neigbours,StartDate,StartHours,EndDate,EndHours
0,6.331585e+08,2015-04-19 01:02:59.000,138.649719,-34.953682,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
1,6.331585e+08,2015-04-19 01:06:59.000,138.649429,-34.953598,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
2,6.331585e+08,2015-04-19 01:09:53.000,138.649429,-34.954014,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
3,6.331585e+08,2015-04-19 01:12:45.000,138.649765,-34.954140,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
4,6.331585e+08,2015-04-19 01:15:37.000,138.649368,-34.954044,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1057315,2.412263e+09,2015-04-07 03:08:57.000,138.644196,-34.837971,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057316,2.412263e+09,2015-04-07 03:19:06.000,138.644257,-34.837906,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057317,2.412263e+09,2015-04-07 03:29:09.000,138.644531,-34.837643,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057318,2.412263e+09,2015-04-07 03:38:05.000,138.644089,-34.837967,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000


In [5]:
classes = df['Hunt']
classes

0          Yes
1          Yes
2          Yes
3          Yes
4          Yes
          ... 
1057315    Yes
1057316    Yes
1057317    Yes
1057318    Yes
1057319    Yes
Name: Hunt, Length: 1057320, dtype: object

We have 2 resulting df : 

* classes consisting of the true Hunt status
* variables consisting of the factors

## Tentative de coller le TP bêtement

In [ ]:
cls = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=1000, 
    per_run_time_limit=120, 
    n_jobs=16,
    include = {"classifier":["random_forest",
                                "lda",
                                "mlp", # multilayer perceptron
                                "gaussian_nb", #  Naive Bayes
                                "k_nearest_neighbors"]},
    memory_limit=24576)

*les param minimum pour la tâche* : 

- time_left_for_this_task= 2000 s
- per_run_time_limit=30 cycles
- n_jobs = 16 coeur
- memory_limit = 24 Go

### Génération des jeux de test, validation

pour notre entrainement, nous prenons des proportions de 67% de test et 33% de test

Les méthodes testées sont : 

* Forêt aléatoire
* Latent Dirichlet Allocation
* Multilayered Perceptron
* Baisien naif
* k plus proches voisins 

### Dans un premier temps, nous allons utiliser seulement les données Quantitatives

#### gestion de la suppression des colonnes quantitatives

In [6]:
dfQuali = variables.drop(["event.id","timestamp","location.long","location.lat",
                          "animal.id","StartDate","StartHours","EndDate","EndHours"],
                         axis = 1).astype('category')

Crée le jeu de test qualitatif

In [ ]:
variables_trainQ, variables_testQ, classes_trainQ, classes_testQ = train_test_split(
        																dfQuali, classes, test_size = 0.33, random_state=0)

Transformation de classesQ en `category` à la place de `object`

In [ ]:
classes_testQ = classes_testQ.astype("category")
classes_trainQ = classes_trainQ.astype("category")

application des paramêtre afin de crée le modèle

In [ ]:
cls.fit(variables_trainQ, classes_trainQ)

Pour l'instant, pn voit que le modèle est claqué :(

	pire que tout, il fonctionne pas 

In [ ]:
cls.leaderboard()

In [ ]:
predictions_Hunt = list(cls.predict(variables_testQ))

précision : 

In [ ]:
print("Accuracy score:", sklearn.metrics.accuracy_score(np.array(classes_testQ), predictions_Hunt))

table des stats

In [ ]:
print( sklearn.metrics.classification_report(classes_testQ, predictions_Hunt) )

décevant : on a un précision catastrophique.

Nous allons regarder la matrice de confusion pour potentiellement observer quel groupe est le mieu prédit

In [ ]:
np.round( confusion_matrix(classes_testQ, predictions_Hunt), 3)

Le problème est clair : on prédit tout en une classe

## Tentative avec du quantitatif car le quali il fonctionne pas encore

### Ouverture du csv

In [7]:
dfQuanti = pd.read_csv('../data/OutCatdataQuantiNormZ.csv', na_filter= False)
dfQuanti = dfQuanti.drop("Unnamed: 0", axis= 1)

/home/jan_codage/miniforge3/envs/fouille/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3552: DtypeWarning: Columns (6,7,8) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


### Suppression des lignes ayant des Na

In [8]:
dfQuanti = dfQuanti[~np.any(dfQuanti == "NA",axis=1)]

### Création des sous jeux de données de test et d'entrainement

In [ ]:
x = dfQuanti.drop('Hunt', axis=1).to_numpy().astype(np.float64)
y = dfQuanti.Hunt.astype(object)

y[y ==	-2.16472428] = "Na"
y[y == 	-0.78922734] = "No"
y[y ==   0.5862696]  = "Yes"

x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size = 0.33, random_state=0)

### Établissement d'un modèle

In [ ]:
cls_hunt = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=900, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["random_forest",
                                "lda",
                                "mlp", # multilayer perceptron
                                "gaussian_nb", #  Naive Bayes
                                "k_nearest_neighbors"]},
    memory_limit=24576)

Ajustment du modèle a notre jeu de données

In [ ]:
cls_hunt.fit(x_train, y_train, dataset_name='Cat Data')

ValueError: Classification with data of type continuous is not supported. Supported types are ['binary', 'multiclass', 'multilabel-indicator']. You can find more information about scikit-learn data types in: https://scikit-learn.org/stable/modules/multiclass.html

## Affichage des résultats

### Affichage des facteurs

In [ ]:
cls_hunt.leaderboard()

NameError: name 'cls' is not defined

### Stockage des prédictions

In [ ]:
predictions_Hunt = list(cls_hunt.predict(x_test))

### Affichage des stats

In [ ]:
print( sklearn.metrics.classification_report(y_test, predictions_Hunt) )

### Affichage de la matrice de confusion

In [ ]:
np.round( confusion_matrix(y_test, predictions_Hunt), 3)

# Prédiction du nombre de proie par chats

## création des 2 matrices

In [ ]:
x2 = dfQuanti.drop('N.pray', axis=1).to_numpy().astype(np.float64)
y2 = df['N.pray'].astype(object)


x2_train, x2_test, y2_train, y2_test = train_test_split(x2, y2,
                                                    test_size = 0.33, random_state=0)

## Configuration du modèle

In [ ]:
cls_Npray = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=900, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["random_forest",
                                "lda",
                                "mlp", # multilayer perceptron
                                "gaussian_nb", #  Naive Bayes
                                "k_nearest_neighbors"]},
    memory_limit=24576)

### Affinage du modèle

In [ ]:
cls_Npray.fit(x2_train, y2_train, dataset_name='Cat Data')

## Affichage des résultats

### Affichage des facteurs

In [ ]:
cls_Npray.leaderboard()

NameError: name 'cls' is not defined

### Stockage des prédictions

In [ ]:
predictions_Npray = list(cls_Npray.predict(x2_test))

### Affichage des stats

In [ ]:
print( sklearn.metrics.classification_report(y2_test, predictions_Npray) )

### Affichage de la matrice de confusion

In [ ]:
np.round( confusion_matrix(y2_test, predictions_Npray), 3)